# ⚕️ HUẤN LUYỆN YOLO11 NHỊ PHÂN PHÁT HIỆN U BUỒNG TRỨNG (MMOTU)
### *Deep Learning for Binary Ovarian Tumor Detection from Ultrasound Images*
---
**Mục tiêu nghiên cứu:**
- Tối ưu hóa độ chính xác phân biệt **Ảnh có bệnh (Tumor)** vs **Ảnh bình thường (Normal)** đạt mức cao nhất.
- Gom toàn bộ 7 nhóm u thành 1 nhãn duy nhất: `0: Tumor` (820 mẫu train, 382 mẫu val).
- Nhóm buồng trứng bình thường (`Normal ovary`) làm mẫu đối chứng âm bản không chứa hộp bao (180 mẫu train, 87 mẫu val).
- Tự động tính toán và xuất các bảng chỉ số y khoa lâm sàng: **Sensitivity, Specificity, Accuracy, F1-Score, mAP@50**.

## 1. Thiết lập Môi trường & Kiểm tra GPU (CUDA)
> **Lưu ý:** Vui lòng chọn menu **Runtime > Change runtime type > T4 GPU** trước khi chạy.

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q ultralytics opencv-python-headless pandas matplotlib seaborn tabulate

import torch
print("=" * 60)
print("🎮 KIỂM TRA CẤU HÌNH PHẦN CỨNG GOOGLE COLAB")
print("=" * 60)
cuda_ok = torch.cuda.is_available()
print(f"CUDA Sẵn sàng: {cuda_ok}")
if cuda_ok:
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dung lượng VRAM: {vram:.2f} GB")
else:
    print("⚠️ Cảnh báo: Đang chạy trên CPU. Hãy vào Runtime > Change runtime type > T4 GPU để tăng tốc!")
print("=" * 60)

## 2. Kết nối Google Drive & Giải nén Bộ dữ liệu MMOTU
Bạn hãy nén thư mục `MMOTU` thành file `MMOTU.zip` và tải lên thư mục gốc của Google Drive (`MyDrive/MMOTU.zip`).

In [ ]:
from google.colab import drive
import zipfile, os, shutil
from pathlib import Path

# Kết nối Google Drive
drive.mount('/content/drive')

# Đường dẫn file zip trên Drive
drive_zip = '/content/drive/MyDrive/MMOTU.zip'
extract_dest = '/content/dataset_raw'

if os.path.exists(drive_zip):
    print(f"Đang giải nén {drive_zip} vào {extract_dest}...")
    os.makedirs(extract_dest, exist_ok=True)
    with zipfile.ZipFile(drive_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_dest)
    print("✅ Giải nén thành công!")
else:
    print(f"⚠️ Không tìm thấy {drive_zip}. Bạn hãy kiểm tra lại đường dẫn file zip trên Drive!")

## 3. Tiền xử lý: Tự động Chuẩn hóa Dữ liệu YOLO Nhị phân (1-Class: Tumor)
- Gom toàn bộ u thành `Class 0: Tumor`.
- Nhóm `Class 5: Normal ovary` làm ảnh nền âm bản (file label rỗng).

In [ ]:
import cv2
import numpy as np
from pathlib import Path

# Tìm thư mục OTU_2d
base_raw = Path('/content/dataset_raw')
mmotu_dirs = list(base_raw.glob('**/OTU_2d'))
if mmotu_dirs:
    mmotu_2d = mmotu_dirs[0]
else:
    mmotu_2d = base_raw / 'OTU_2d'

images_dir = mmotu_2d / 'images'
annotations_dir = mmotu_2d / 'annotations'

output_dir = Path('/content/dataset_binary')
train_img_dir = output_dir / 'images' / 'train'
val_img_dir = output_dir / 'images' / 'val'
train_lbl_dir = output_dir / 'labels' / 'train'
val_lbl_dir = output_dir / 'labels' / 'val'

def load_cls(fpath):
    mapping = {}
    if not os.path.exists(fpath): return mapping
    with open(fpath, 'r', encoding='utf-8') as f:
        for line in f:
            p = line.strip().split()
            if len(p) >= 2:
                mapping[p[0]] = int(p[1])
    return mapping

def extract_bboxes(mask_path, img_w, img_h, min_area=30):
    if not os.path.exists(mask_path): return []
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return []
    thresh = (mask > 0).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in cnts:
        if cv2.contourArea(c) < min_area: continue
        x, y, w, h = cv2.boundingRect(c)
        xc = min(1.0, max(0.0, (x + w / 2.0) / img_w))
        yc = min(1.0, max(0.0, (y + h / 2.0) / img_h))
        norm_w = min(1.0, max(0.0, w / float(img_w)))
        norm_h = min(1.0, max(0.0, h / float(img_h)))
        boxes.append((xc, yc, norm_w, norm_h))
    return boxes

def convert_split(cls_map, target_img, target_lbl, split_name='train'):
    os.makedirs(target_img, exist_ok=True)
    os.makedirs(target_lbl, exist_ok=True)
    stats = {'Normal': 0, 'Tumor': 0, 'Boxes': 0}
    for img_name, cls_id in cls_map.items():
        src_img = images_dir / img_name
        if not src_img.exists(): continue
        shutil.copy2(src_img, target_img / img_name)
        img = cv2.imread(str(src_img))
        if img is None: continue
        h, w = img.shape[:2]
        
        base_id = Path(img_name).stem
        m_path = annotations_dir / f"{base_id}_binary.PNG"
        if not m_path.exists(): m_path = annotations_dir / f"{base_id}.PNG"
        
        bboxes = extract_bboxes(m_path, w, h)
        lbl_file = target_lbl / f"{base_id}.txt"
        with open(lbl_file, 'w', encoding='utf-8') as lf:
            if cls_id == 5:
                stats['Normal'] += 1
            else:
                for xc, yc, bw, bh in bboxes:
                    lf.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
                    stats['Boxes'] += 1
                stats['Tumor'] += 1
    print(f"[{split_name.upper()}] Có bệnh (Tumor): {stats['Tumor']} ảnh ({stats['Boxes']} boxes) | Không bệnh (Normal): {stats['Normal']} ảnh")
    return stats

train_map = load_cls(mmotu_2d / 'train_cls.txt')
val_map = load_cls(mmotu_2d / 'val_cls.txt')
print(f"Tìm thấy {len(train_map)} mẫu train, {len(val_map)} mẫu val.")

convert_split(train_map, train_img_dir, train_lbl_dir, 'train')
convert_split(val_map, val_img_dir, val_lbl_dir, 'val')

# Tạo dataset.yaml
yaml_content = f"""path: /content/dataset_binary
train: images/train
val: images/val
names:
  0: Tumor
"""
with open(output_dir / 'dataset.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content)
print("✅ Đã tạo xong cấu hình dataset_binary.yaml!")

## 4. Huấn luyện Mô hình YOLO11 Nhị phân trên GPU T4
- Base weights: `yolo11n.pt` (hoặc `yolo11s.pt`).
- Epochs: 30 | Batch: 16 | Imgsz: 640 | Optimizer: AdamW.

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình YOLO11n
model = YOLO('yolo11n.pt')

print("Bắt đầu huấn luyện YOLO11 Nhị phân trên GPU...")
results = model.train(
    data='/content/dataset_binary/dataset.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=12,
    project='/content/runs_binary',
    name='ovarian_tumor_binary',
    plots=True,
    verbose=True
)

print("✅ Huấn luyện hoàn tất! Trọng số lưu tại /content/runs_binary/ovarian_tumor_binary/weights/best.pt")

## 5. Đánh giá Chuyên sâu & Tính toán Ma trận Nhầm lẫn Nhị phân (TP, FP, TN, FN)
Kiểm thử trên toàn bộ 469 ảnh tập Validation để đo độ chính xác phân loại **Có bệnh vs Không bệnh**.

In [ ]:
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# Load model best
best_weights = '/content/runs_binary/ovarian_tumor_binary/weights/best.pt'
model_eval = YOLO(best_weights)

# Chạy đánh giá validation chuẩn của YOLO
val_metrics = model_eval.val(data='/content/dataset_binary/dataset.yaml', split='val')
map50 = val_metrics.box.map50
map50_95 = val_metrics.box.map

# Quét toàn bộ tập Validation ở ngưỡng confidence để tính Ma trận nhầm lẫn Y khoa
val_img_dir = Path('/content/dataset_binary/images/val')
val_lbl_dir = Path('/content/dataset_binary/labels/val')

CONF_THRESH = 0.35 # Ngưỡng tin cậy tối ưu
TP, FP, TN, FN = 0, 0, 0, 0

val_images = list(val_img_dir.glob('*.JPG'))
print(f"Đang đánh giá trên {len(val_images)} ảnh validation...")

for img_p in val_images:
    base_name = img_p.stem
    lbl_p = val_lbl_dir / f"{base_name}.txt"
    
    # Xác định Ground Truth: file có nội dung là Có bệnh (1), file rỗng là Không bệnh (0)
    is_gt_diseased = False
    if lbl_p.exists():
        with open(lbl_p, 'r') as f:
            if len(f.read().strip()) > 0:
                is_gt_diseased = True
                
    # Dự đoán với model
    preds = model_eval.predict(source=str(img_p), conf=CONF_THRESH, verbose=False)
    is_pred_diseased = False
    if len(preds) > 0 and preds[0].boxes is not None and len(preds[0].boxes) > 0:
        is_pred_diseased = True
        
    if is_gt_diseased and is_pred_diseased:
        TP += 1
    elif not is_gt_diseased and is_pred_diseased:
        FP += 1
    elif not is_gt_diseased and not is_pred_diseased:
        TN += 1
    elif is_gt_diseased and not is_pred_diseased:
        FN += 1

# Tính các chỉ số lâm sàng
sensitivity = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0
specificity = (TN / (TN + FP)) * 100 if (TN + FP) > 0 else 0
accuracy = ((TP + TN) / (TP + TN + FP + FN)) * 100
precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0
f1 = (2 * precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

print("✅ Đã tính toán xong toàn bộ chỉ số!")

## 6. Xuất Bảng Kết Quả Trực quan (Clinical Metrics Table)
Hiển thị bảng kết quả định dạng chuyên nghiệp dưới dạng HTML styling trực tiếp trên Colab.

In [ ]:
from IPython.display import display, HTML

# Bảng ma trận nhầm lẫn 2x2
cm_data = {
    'Thực tế (Ground Truth)': ['Có bệnh (Khối u)', 'Bình thường (Không u)', 'Tổng cộng'],
    'Dự đoán Có bệnh': [f"{TP} (TP - Đoán trúng)", f"{FP} (FP - Báo động giả)", f"{TP + FP}"],
    'Dự đoán Không bệnh': [f"{FN} (FN - Bỏ sót u)", f"{TN} (TN - Chuẩn âm tính)", f"{FN + TN}"],
    'Tổng thực tế': [f"{TP + FN}", f"{FP + TN}", f"{TP + TN + FP + FN}"]
}
df_cm = pd.DataFrame(cm_data)

# Bảng chỉ số y khoa
metrics_data = {
    'Chỉ số Đánh giá (Metrics)': [
        'Độ chính xác xác nhận Bệnh / Không bệnh (Accuracy)',
        'Độ nhạy / Bắt trúng ca bệnh (Sensitivity / Recall)',
        'Độ đặc hiệu / Phân biệt người bình thường (Specificity)',
        'Độ tin cậy dự đoán (Precision)',
        'Điểm F1-Score cân bằng',
        'Độ chính xác định vị hộp bao u (mAP@50)',
        'Độ chính xác định vị nghiêm ngặt (mAP@50-95)'
    ],
    'Công thức Tính': [
        '(TP + TN) / Total',
        'TP / (TP + FN)',
        'TN / (TN + FP)',
        'TP / (TP + FP)',
        '2 * (P * R) / (P + R)',
        'Area Under PR Curve @ IoU 0.5',
        'Average mAP @ IoU 0.5:0.95'
    ],
    'Kết quả Đạt được': [
        f"{accuracy:.2f}% ✨",
        f"{sensitivity:.2f}%✅",
        f"{specificity:.2f}%✅",
        f"{precision:.2f}%",
        f"{f1:.2f}%",
        f"{map50 * 100:.2f}% ⭐",
        f"{map50_95 * 100:.2f}%"
    ]
}
df_metrics = pd.DataFrame(metrics_data)

html_out = f"""
<div style='font-family: Arial, sans-serif; max-width: 900px; margin: 20px 0;'>
  <h3 style='color: #0284c7; border-bottom: 2px solid #0284c7; padding-bottom: 8px;'>🎯 BẢNG 1: MA TRẬN NHẦM LẪN NHỊ PHÂN (CONFUSION MATRIX 2x2)</h3>
  {df_cm.to_html(index=False, classes='table table-bordered')}
  
  <h3 style='color: #16a34a; border-bottom: 2px solid #16a34a; padding-bottom: 8px; margin-top: 30px;'>🏆 BẢNG 2: BẢNG TỔNG HỢP CÁC CHỈ SỐ LÂM SÀNG Y KHOA</h3>
  {df_metrics.to_html(index=False, classes='table table-hover')}
</div>
"""
display(HTML(html_out))

# Lưu thành file CSV
df_metrics.to_csv('/content/runs_binary/clinical_metrics_summary.csv', index=False, encoding='utf-8-sig')
print("Đã lưu bảng kết quả thành file: /content/runs_binary/clinical_metrics_summary.csv")

## 7. Hiển thị các Biểu đồ Hiệu năng & Dự đoán Mẫu trực quan
Hiển thị các biểu đồ Precision-Recall, F1-Curve và ảnh test so sánh Ground Truth vs Prediction.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = Path('/content/runs_binary/ovarian_tumor_binary')
fig_files = ['results.png', 'BoxPR_curve.png', 'BoxF1_curve.png', 'confusion_matrix.png']

for f in fig_files:
    p = run_dir / f
    if p.exists():
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(str(p))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f, fontsize=14, fontweight='bold')
        plt.show()

# Hiển thị 3 ảnh dự đoán mẫu trên tập validation
sample_preds = model_eval.predict(source=str(val_img_dir), conf=0.35, max_det=5, save=False)
print("Hiển thị kết quả phát hiện khối u trên mẫu ảnh thực tế:")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i in range(min(3, len(sample_preds))):
    im_bgr = sample_preds[i].plot()
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(im_rgb)
    axes[i].axis('off')
    axes[i].set_title(f"Mẫu {i+1}", fontsize=12)
plt.tight_layout()
plt.show()

## 8. Tự động Sao lưu Trọng số và Kết quả về Google Drive
Lưu file `best.pt` và bảng chỉ số CSV vào thư mục `MyDrive/NCKH_YOLO11_Binary_Results/`.

In [ ]:
backup_dir = Path('/content/drive/MyDrive/NCKH_YOLO11_Binary_Results')
os.makedirs(backup_dir, exist_ok=True)

# Copy model best.pt
if os.path.exists(best_weights):
    shutil.copy2(best_weights, backup_dir / 'best_binary.pt')
    print(f"✅ Đã sao lưu trọng số: {backup_dir}/best_binary.pt")

# Copy các file kết quả
for f in ['clinical_metrics_summary.csv', 'results.csv', 'results.png', 'BoxPR_curve.png', 'confusion_matrix.png']:
    src = run_dir / f if (run_dir / f).exists() else Path('/content/runs_binary') / f
    if src.exists():
        shutil.copy2(src, backup_dir / f)

print(f"🎉 Tất cả kết quả đã được sao lưu an toàn về Google Drive tại: {backup_dir}")